# RNN/LSTM Image Captioning

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src" / "rnn").exists()), cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

## Configuration

In [ ]:
from src.rnn.paths import RnnPaths

paths = RnnPaths.from_root(REPO_ROOT)
SEQ_LEN = 35
EVAL_LIMIT = 100
EPOCHS = 5
BATCH_SIZE = 64
RUN_TRAINING = False
FORCE_FEATURE_EXTRACTION = False

## Feature Extraction

In [ ]:
from src.rnn.feature_extraction import extract_and_save_repo_features
from src.rnn.experiment import load_feature_map

features_exist = paths.features_file.exists() and paths.feature_names_file.exists()
if FORCE_FEATURE_EXTRACTION or not features_exist:
    features_matrix, image_names = extract_and_save_repo_features(
        REPO_ROOT,
        batch_size=32,
        force=FORCE_FEATURE_EXTRACTION,
    )
else:
    print(f"Feature cache found: {paths.feature_dir}")

image_features_map = load_feature_map(REPO_ROOT)
print(f"Loaded {len(image_features_map)} image feature vectors")

## Caption Data

In [ ]:
from src.rnn.experiment import load_caption_sequences, load_text_util
from src.rnn.training import prepare_training_data

text_util = load_text_util(REPO_ROOT, sequence_length=SEQ_LEN)
caption_mapping, sequence_mapping = load_caption_sequences(REPO_ROOT, text_util)
training_data = prepare_training_data(REPO_ROOT, sequence_length=SEQ_LEN)

print(f"Vocabulary size: {text_util.vocab_size}")
print(f"Captions mapped: {len(caption_mapping)} images")
print(f"Train/val/test: {len(training_data['train_keys'])}/{len(training_data['val_keys'])}/{len(training_data['test_keys'])}")

## Keras Training

In [ ]:
from src.rnn.experiment import load_training_history
from src.rnn.training import train_all_variations

if RUN_TRAINING:
    history_df = train_all_variations(
        REPO_ROOT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        sequence_length=SEQ_LEN,
        force=True,
    )
else:
    history_df = load_training_history(REPO_ROOT)

history_df[["model_type", "variation_name", "layers", "hidden_state", "final_loss", "final_val_loss", "training_time_sec"]]

## Scratch Caption Smoke Test

In [ ]:
from src.rnn.ImageCaptioningScratch import ImageCaptioningModel
from src.rnn.experiment import make_keras_model

keras_model = make_keras_model(REPO_ROOT, text_util, "LSTM", "Shallow_Small", 1, 128)
scratch_model = ImageCaptioningModel(keras_model, text_util, is_lstm=True)

sample_image = training_data["test_keys"][0]
sample_caption = scratch_model.generate_caption(image_features_map[sample_image], max_len=10)
print(sample_image)
print(sample_caption)

## Full Evaluation

In [ ]:
from src.rnn.experiment import (
    compare_best_keras_vs_scratch,
    evaluate_all_variations,
    max_length_sweep,
    qualitative_samples,
    write_analysis_summary,
)

In [ ]:
variation_results, caption_details = evaluate_all_variations(
    REPO_ROOT,
    split="test",
    limit=EVAL_LIMIT,
    max_len=SEQ_LEN,
)
variation_results.sort_values(["model_type", "scratch_bleu_4"], ascending=[True, False])

In [ ]:
keras_vs_scratch = compare_best_keras_vs_scratch(
    REPO_ROOT,
    split="test",
    limit=EVAL_LIMIT,
    max_len=SEQ_LEN,
)
keras_vs_scratch

In [ ]:
max_length_results = max_length_sweep(
    REPO_ROOT,
    lengths=(10, 20, 35),
    split="test",
    limit=EVAL_LIMIT,
)
max_length_results

In [ ]:
samples = qualitative_samples(REPO_ROOT, limit=EVAL_LIMIT, n_per_bucket=4)
samples

In [ ]:
summary_path = write_analysis_summary(REPO_ROOT)
print(f"Ringkasan analisis disimpan di: {summary_path}")

## Training Curves

In [ ]:
from src.rnn.visualization import plot_training_history

plot_training_history(REPO_ROOT)